## This notebook runs the end-to-end FluRICo pipeline

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from core.flurico import FluRiCoAnalysis


### Data inputs

In [3]:
df_mb = pd.read_csv('data/microbe_train.csv',index_col=0)
df_mt = pd.read_csv('data/metabolite_train.csv',index_col=0)

In [6]:
df_mb = np.log1p(df_mb)
df_mt = np.log1p(df_mt)

In [ ]:

df_mb_corr = {}
for i in ['HC','MIA','IA']:
    df_mb_corr[i] = pd.read_csv(f'results/fastspar/{i}_median_correlation_5146.tsv',sep='\t',index_col=0)
    

### FluRICo process

In [8]:

analyzer = FluRiCoAnalysis(microbe_data=df_mb, metabolite_data=df_mt, 
                           group_labels=['HC','MIA','IA'], microbe_coab_data = df_mb_corr)

In [9]:
all_scores = analyzer.calculate_all_scores()

==== Starting full FluRiCo score computation ====
Computing Fluctuation...
  [microbe] Computing Variance Heterogeneity and Change Effect scores...


  Levene test (multi-group): 100%|██████████| 5146/5146 [00:06<00:00, 793.58it/s]


  [metabolite] Computing Variance Heterogeneity and Change Effect scores...


  Levene test (multi-group): 100%|██████████| 763/763 [00:00<00:00, 835.09it/s]


Computing Intra-omics Correlation...
  [microbe] Group HC: computing Intra-omics Correlation from the co-abundance matrix...
  [metabolite] Group HC: computing metabolite self-correlation for Intra-omics Correlation...
  [microbe] Group MIA: computing Intra-omics Correlation from the co-abundance matrix...
  [metabolite] Group MIA: computing metabolite self-correlation for Intra-omics Correlation...
  [microbe] Group IA: computing Intra-omics Correlation from the co-abundance matrix...
  [metabolite] Group IA: computing metabolite self-correlation for Intra-omics Correlation...
Computing Cross-omics Correlation...
  [microbe] Group HC: microbe -> metabolite Cross-omics Correlation...


...

  [microbe] Group IA: microbe -> metabolite Cross-omics Correlation...


IA - microbe: 100%|██████████| 5146/5146 [04:43<00:00, 18.18it/s]


  [metabolite] Group IA: metabolite -> microbe Cross-omics Correlation...


IA - metabolite: 100%|██████████| 763/763 [00:57<00:00, 13.25it/s]


Computing Remote Trend...
  [step 1/4] Checking whether Change Effect Detail is available...
  Detected 3 labels; using the cosine-similarity method...
  [step 2/4] Building trend vectors...
  [step 3/4] Computing the cosine similarity matrix...
    Cosine matrix shape: 5146 microbes x 763 metabolites
  [step 4/4] Counting Positive Links and Negative Links...
  Remote Trend computation completed.
==== Full FluRiCo score computation completed ====


In [10]:
analyzer.save_results(output_dir='results/flurico_results/')

microbe, summary scores for group HC were saved to results/flurico_results/microbe_summary_HC.csv
microbe, summary scores for group MIA were saved to results/flurico_results/microbe_summary_MIA.csv
microbe, summary scores for group IA were saved to results/flurico_results/microbe_summary_IA.csv
microbe Variance Heterogeneity p/FDR values were saved to results/flurico_results/microbe_variance_heterogeneity_p_values.csv
microbe Change Effect details were saved to results/flurico_results/microbe_change_effect_detail.csv
metabolite, summary scores for group HC were saved to results/flurico_results/metabolite_summary_HC.csv
metabolite, summary scores for group MIA were saved to results/flurico_results/metabolite_summary_MIA.csv
metabolite, summary scores for group IA were saved to results/flurico_results/metabolite_summary_IA.csv
metabolite Variance Heterogeneity p/FDR values were saved to results/flurico_results/metabolite_variance_heterogeneity_p_values.csv
metabolite Change Effect detail

### Result reload

In [17]:
score_hc_mb = pd.read_csv('results/flurico_results/microbe_summary_HC.csv',index_col=0)
score_mia_mb = pd.read_csv('results/flurico_results/microbe_summary_MIA.csv',index_col=0)
score_ia_mb = pd.read_csv('results/flurico_results/microbe_summary_IA.csv',index_col=0)

score_hc_mt = pd.read_csv('results/flurico_results/metabolite_summary_HC.csv',index_col=0)
score_mia_mt = pd.read_csv('results/flurico_results/metabolite_summary_MIA.csv',index_col=0)
score_ia_mt = pd.read_csv('results/flurico_results/metabolite_summary_IA.csv',index_col=0)

In [18]:
score_hc_mb

,Fluctuation_Levene,Fluctuation_FC_HC_vs_MIA,Fluctuation_FC_MIA_vs_IA,Fluctuation_FC_HC_vs_IA,Coordination,Remote_Interaction,Remote_Trend_Pos_Count,Remote_Trend_Neg_Count
g__Desulfuromonas;s__Desulfuromonas acetexigens,1.000000e+00,0.263817,0.040398,0.000000,0.101870,0.161970,0.060837,0.416031
g__Sporosarcina;s__Sporosarcina sp. P18a,8.884658e-01,0.023204,0.020864,0.437397,0.442620,0.009695,0.330798,0.106870
g__Verrucosispora;s__Verrucosispora sediminis,8.802620e-01,0.647767,0.034875,0.000000,0.137135,0.237427,0.422053,0.080153
g__Actinomyces;s__Actinomyces hordeovulneris,8.353200e-01,0.048656,0.127842,0.322642,0.037133,0.150866,0.178707,0.332061
g__Unclassified;s__Deltaproteobacteria bacterium RBG_13_53_10,8.142404e-01,0.221394,0.077004,0.124257,0.242659,0.145781,0.361217,0.874046
...,...,...,...,...,...,...,...,...
g__Alkalibacterium;s__Alkalibacterium gilvum,3.810631e-09,0.001241,0.004784,0.002408,0.259340,0.001345,0.467681,0.587786
g__Alkaliphilus;s__Alkaliphilus metalliredigens,3.113208e-09,0.021296,0.064229,0.004119,0.682891,0.112361,0.444867,0.217557
g__Flexibacter;s__Flexibacter flexilis,2.142825e-09,0.000685,0.002671,0.000000,0.355368,0.033149,0.429658,0.202290
g__Bacillus;s__Bacillus sp. J13,1.628258e-09,0.003995,0.194335,0.029095,0.312230,0.056425,0.433460,0.229008


### Feature prioritization

In [24]:
def find_top_features(df, top_n=50, top_n_dic=None):
    if top_n_dic is None:
        top_n_dic = {}

    dim_map = {
        'flu_le': 'Fluctuation_Levene',
        'flu_fc_ea': 'Fluctuation_FC_HC_vs_MIA',
        'flu_fc_la': 'Fluctuation_FC_MIA_vs_IA',
        'flu_fc_gl': 'Fluctuation_FC_HC_vs_IA',
        'ri': 'Remote_Interaction',
        'ri_tr_pos_c': 'Remote_Trend_Pos_Count',
        'ri_tr_neg_c': 'Remote_Trend_Neg_Count',
        'co': 'Coordination'
    }

    result = {}
    for key, col in dim_map.items():
        n = top_n_dic.get(key, top_n)
        result[key] = df.nlargest(n, col, keep='all')[col].to_dict()

    return result


def ordered_union(*lists):
    seen = set()
    out = []
    for lst in lists:
        for x in lst:
            if x not in seen:
                seen.add(x)
                out.append(x)
    return out


def build_feature_lists(score_hc, score_mia, score_ia, top_n):
    rank_dict = {
        'hc': find_top_features(score_hc, top_n=top_n),
        'mia': find_top_features(score_mia, top_n=top_n),
        'ia': find_top_features(score_ia, top_n=top_n)
    }

    union_dict = {
        dim: ordered_union(
            list(rank_dict['hc'][dim].keys()),
            list(rank_dict['mia'][dim].keys()),
            list(rank_dict['ia'][dim].keys())
        )
        for dim in rank_dict['hc']
    }

    dy_list = ordered_union(
        union_dict['flu_le'],
        union_dict['flu_fc_ea'],
        union_dict['flu_fc_la'],
        union_dict['flu_fc_gl'],
        union_dict['ri_tr_pos_c'],
        union_dict['ri_tr_neg_c']
    )

    st_list = ordered_union(
        union_dict['ri'],
        union_dict['co']
    )

    return rank_dict, union_dict, dy_list, st_list

In [25]:
# Microbe
mb_rank, mb_union, mb_dy_list, mb_st_list = build_feature_lists(
    score_hc_mb, score_mia_mb, score_ia_mb, top_n=50
)


# Metabolite
mt_rank, mt_union, mt_dy_list, mt_st_list = build_feature_lists(
    score_hc_mt, score_mia_mt, score_ia_mt, top_n=20
)


In [30]:
len(mb_dy_list),len(mb_st_list),len(mt_dy_list),len(mt_st_list),

(281, 249, 109, 99)